In [ ]:
import pandas as pd
from loguru import logger
from data_preprocessing.utils.data_handling import load_parquet_table
from pathlib import Path

from tqdm.auto import tqdm

import torch
import torch.nn as nn
from playground.time_series_panel_dataloader import TimeSeriesPanelDataset
from deep_learning.LSTM import ShallowRegressionLSTM

In [ ]:
notebook_path = Path.cwd()
project_root = notebook_path.parents[0]
dataset_ts_path: Path = project_root / Path(
    "datasets/raw_and_shitty/dataset_ts.parquet")

In [ ]:
df_dataset_ts = load_parquet_table(parquet_path=dataset_ts_path, verbose=False)
df_dataset_ts.head()

In [ ]:
df_dataset_ts.describe()

https://www.crosstab.io/articles/time-series-pytorch-lstm/#the-model-and-learning-algorithm

In [ ]:
patient_ids = df_dataset_ts["patientunitstayid"].unique().tolist()
df_dataset_ts_reduced = df_dataset_ts.loc[:,
                      ["patientunitstayid", "time_offset", "heartrate", "sao2", "temperature", "st1", "sepsis_present"]]
# df_dataset_cont = df_dataset_cont_all[df_dataset_cont_all["patientunitstayid"].isin(patient_ids[0:29])].copy()
df_dataset_ts_reduced = df_dataset_ts_reduced[df_dataset_ts_reduced["patientunitstayid"].isin(patient_ids)].copy()


In [ ]:
df_dataset_ts_sepsis = df_dataset_ts[df_dataset_ts["sepsis_patient"] == True]
patient_ids_sepsis = df_dataset_ts_sepsis["patientunitstayid"].unique().tolist()

In [ ]:
target = "sepsis_present"
features = list(df_dataset_ts_reduced.columns.difference([target]))

In [ ]:
df_train = df_dataset_ts_reduced.loc[df_dataset_ts_reduced["patientunitstayid"].isin(patient_ids[0:100])]
df_test = df_dataset_ts_reduced.loc[df_dataset_ts_reduced["patientunitstayid"].isin(patient_ids[101:200])]
len(df_test) / len(df_train)

# Dataloader


In [ ]:
df_train.head()

dataset_train_ids = []

#for idx, val in enumerate(df_train[["patientunitstayid", "time_offset"]]):

#    patient_id_for_dataset_id = df_train["patientunitstayid"]
#    time_offset_for_dataset_id = df_train[("time_offset")]

#    dataset_train_ids.append([patient_id_for_dataset_id, time_offset_for_dataset_id])


df_dataset_ids = list(zip(df_train["patientunitstayid"], df_train["time_offset"]))



In [ ]:
i = 1
sequence_length = 4

train_dataset = TimeSeriesPanelDataset(df_train, features, target)
len(train_dataset)

# X, y = train_dataset[i]
features_sample, target_sample = train_dataset[200]

#features_sample

In [ ]:
#target_sample

In [ ]:
def collate_pad_sequences(batch):
    logger.debug(f"{batch[0] = }")
    lengths = torch.tensor([sample[0].shape[0] for sample in batch])  #.to(device)

    #batch = [torch.Tensor(t) for t in batch] # [ torch.Tensor(t).to(device) for t in batch ]
    #batch = [*t for t in batch] # [ torch.Tensor(t).to(device) for t in batch ]
    logger.debug(f"Batch to pad {batch[0] = }")
    batch = torch.nn.utils.rnn.pack_padded_sequence(batch)
    mask = (batch != 0)  #.to(device)
    return batch, lengths, mask


def collate_pack_padded_sequences(batch):
    # features_to_collate, target_to_collate = batch
    lengths_of_batch_items = [item[0].shape[0] for item in batch] 
    
    x_batch = batch[0]
    
    pack_padded_x = torch.nn.utils.rnn.pack_padded_sequence(x_batch, lengths=lengths_of_batch_items)    
    
    return pack_padded_x

def collate_fn(batch):
    batch = [sample for sample in batch]
    return torch.utils.data.dataloader.default_collate(batch)

for item in batch_test:
    # getting the number rows x features of the features
    # batch returns a tuple of (features, targets)
    # torch.Size([2, 4]) -> shape[0] -> [2]
   
    # logger.debug(item[0].shape)
    logger.debug(item[0].shape[0])



len_batch_items = [len(item[0]) for item in batch_test]

torch.nn.utils.rnn.pack_padded_sequence(input=batch_test, lengths=len_batch_items)

In [ ]:
max_length = torch.tensor(data=[4, 3, 2, 1])

# Datset from the Dataloader

In [ ]:
from torch.utils.data import DataLoader

train_dataset = TimeSeriesPanelDataset(dataframe=df_train, target=target, features=features)
test_dataset = TimeSeriesPanelDataset(dataframe=df_test, target=target, features=features)

# train_loader = DataLoader(train_dataset, batch_size=10, shuffle=True, collate_fn=collate_pack_padded_sequences)
train_loader = DataLoader(train_dataset, batch_size=1, shuffle=True)
# test_loader = DataLoader(test_dataset, batch_size=10, shuffle=False, collate_fn=collate_pack_padded_sequences)
test_loader = DataLoader(test_dataset, batch_size=1, shuffle=False)



In [ ]:
test_sample = next(iter(test_loader))

In [ ]:

device: torch.device = (
    torch.device("cuda") if torch.cuda.is_available() else torch.device("cpu")
)
logger.info(f"Training device/hardware = {device} ")


In [ ]:
learning_rate = 5e-5
num_hidden_units = 16
num_features = 4 

model = ShallowRegressionLSTM(num_features=num_features, hidden_size=num_hidden_units)

# loss_fn = nn.MSELoss()
#loss_fn = nn.CrossEntropyLoss(weight=torch.tensor(class_weights[::-1])).to(device=device)
loss_fn = nn.CrossEntropyLoss().to(device=device)
optimizer = torch.optim.Adam(model.parameters(), lr=learning_rate)

In [ ]:
def train_model(train_dataloader: torch.utils.data.DataLoader, model: torch.nn.Module, loss_function, optimizer):
    num_batches = len(train_loader)
    total_loss = 0
    model = model.to(device=device)
    model.train()

    batch_progress_bar = tqdm(enumerate(train_dataloader), total=len(train_dataloader), unit="batch", desc="Training Batch")
    
    for X, y in train_dataloader:
    # for X, y in batch_progress_bar:
        X = X.to(device=device)
        labels = y.squeeze(0)
        entire_sequence_label = labels.argmax(dim=0).to(device=device)
        
        output = model(X)
        # loss = loss_function(output, y)
        loss = loss_function(output, entire_sequence_label)

        optimizer.zero_grad()
        loss.backward()
        optimizer.step()

        total_loss += loss.item()
        
        #logger.info(f"{total_loss = }")

    avg_loss = total_loss / num_batches

    logger.info(f"Training Loss: {avg_loss:.4f}")


def test_model(test_dataloader: torch.utils.data.DataLoader, model: torch.nn.Module, loss_function):
    num_batches = len(test_loader)
    total_loss = 0


    model = model.to(device=device)
    model.eval()

    with torch.no_grad():
        for X, y in test_dataloader:
            labels = y.squeeze(0)
            # Todo: This is just for testing purposes, i.e. checking if there's any sepsis present label
            # Todo (cont): present in the dataset for this chunk of the sequence, 
            # Todo (cont): this means that we make just a classification of the ts, instead of pinpointing it
            entire_sequence_label = labels.argmax(dim=0).to(device=device)
            
            output = model(X.to(device=device))
            total_loss += loss_function(output, entire_sequence_label).item()
            #logger.info(f"{total_loss = }")

    avg_loss = total_loss / num_batches

    logger.info(f"Test Loss: {avg_loss:.4f}")


logger.info("Untrained test \n")
#test_model(test_dataloader=test_loader, model=model, loss_function=loss_fn)

In [ ]:
for ix_epoch in range(2):
    print(f"Epoch {ix_epoch}\n---------")
    train_model(train_loader, model, loss_fn, optimizer=optimizer)
    test_model(test_loader, model, loss_fn)